# Checkpoints

PathSim supports saving and loading simulation state via checkpoints. This allows you to pause a simulation, save its complete state to disk, and resume it later from exactly where you left off.

Checkpoints also enable **rollback** — returning to a saved state and exploring different what-if scenarios by changing parameters.

Checkpoints use a split format: a JSON file for metadata and structure, and an NPZ file for numerical data (block states, solver histories, etc.).

## Building the System

We'll use the coupled oscillators system to demonstrate checkpoints. The energy exchange between the two oscillators produces a sustained, non-trivial response that makes it easy to visually verify checkpoint continuity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import ODE, Function, Scope

Define the system parameters:

In [ ]:
# Mass parameters
m1 = 1.0
m2 = 1.5

# Spring constants
k1 = 2.0
k2 = 3.0
k12 = 0.5  # coupling spring

# Damping coefficients
c1 = 0.02
c2 = 0.03

# Initial conditions [position, velocity]
x1_0 = np.array([2.0, 0.0])  # oscillator 1 displaced
x2_0 = np.array([0.0, 0.0])  # oscillator 2 at rest

In [ ]:
# Oscillator 1: m1*x1'' = -k1*x1 - c1*x1' - k12*(x1 - x2)
def osc1_func(x1, u, t):
    f_e = u[0]
    return np.array([x1[1], (-k1*x1[0] - c1*x1[1] - f_e) / m1])

# Oscillator 2: m2*x2'' = -k2*x2 - c2*x2' + k12*(x1 - x2)
def osc2_func(x2, u, t):
    f_e = u[0]
    return np.array([x2[1], (-k2*x2[0] - c2*x2[1] - f_e) / m2])

# Coupling force
def coupling_func(x1, x2):
    f = k12 * (x1 - x2)
    return f, -f

# Blocks
osc1 = ODE(osc1_func, x1_0)
osc2 = ODE(osc2_func, x2_0)
fn   = Function(coupling_func)
sc   = Scope(labels=[r"$x_1(t)$ - Oscillator 1", r"$x_2(t)$ - Oscillator 2"])

blocks = [osc1, osc2, fn, sc]

# Connections
connections = [
    Connection(osc1[0], fn[0], sc[0]),
    Connection(osc2[0], fn[1], sc[1]),
    Connection(fn[0], osc1[0]),
    Connection(fn[1], osc2[0]),
]

In [ ]:
sim = Simulation(blocks, connections, dt=0.01)

sim.run(60)

fig, ax = sc.plot()
plt.show()

The two oscillators exchange energy through the coupling spring, producing a characteristic beat pattern.

## Saving a Checkpoint

Now let's save the simulation state at t=60s. This creates two files: `coupled.json` (metadata) and `coupled.npz` (numerical data).

In [ ]:
sim.save_checkpoint("coupled")
print(f"Checkpoint saved at t = {sim.time:.1f}s")

We can inspect the JSON file to see what was saved:

In [ ]:
import json

with open("coupled.json") as f:
    cp = json.load(f)

print(f"PathSim version: {cp['pathsim_version']}")
print(f"Simulation time: {cp['simulation']['time']:.1f}s")
print(f"Solver: {cp['simulation']['solver']}")
print(f"Blocks saved:")
for b in cp["blocks"]:
    print(f"  {b['_key']} ({b['type']})")

## Rollback: What-If Scenarios

This is where checkpoints really shine. We'll load the same checkpoint three times with different coupling strengths to explore how the system evolves from the exact same state.

Since the checkpoint restores all block states by type and insertion order, we just need to rebuild the simulation with the same block structure but different parameters.

In [ ]:
def run_scenario(k12_new, duration=60):
    """Load checkpoint and continue with a different coupling constant."""
    def coupling_new(x1, x2):
        f = k12_new * (x1 - x2)
        return f, -f

    o1 = ODE(osc1_func, x1_0)
    o2 = ODE(osc2_func, x2_0)
    f  = Function(coupling_new)
    s  = Scope()

    sim = Simulation(
        [o1, o2, f, s],
        [Connection(o1[0], f[0], s[0]),
         Connection(o2[0], f[1], s[1]),
         Connection(f[0], o1[0]),
         Connection(f[1], o2[0])],
        dt=0.01
    )
    sim.load_checkpoint("coupled")
    sim.run(duration)
    return s.read()

# Original coupling (continuation)
t_a, d_a = run_scenario(k12_new=0.5)

# Stronger coupling
t_b, d_b = run_scenario(k12_new=2.0)

# Decoupled
t_c, d_c = run_scenario(k12_new=0.0)

## Comparing the Scenarios

The plot shows the original run (0-60s) followed by three different futures branching from the checkpoint at t=60s. We show oscillator 1 for clarity.

In [ ]:
time_orig, data_orig = sc.read()

fig, ax = plt.subplots(figsize=(10, 4))

# Original run
ax.plot(time_orig, data_orig[0], "k-", lw=1.5, label=r"original ($k_{12}=0.5$)")

# Three futures from checkpoint
ax.plot(t_a, d_a[0], "C0-", alpha=0.8, label=r"continued ($k_{12}=0.5$)")
ax.plot(t_b, d_b[0], "C1-", alpha=0.8, label=r"stronger coupling ($k_{12}=2.0$)")
ax.plot(t_c, d_c[0], "C2-", alpha=0.8, label=r"decoupled ($k_{12}=0$)")

ax.axvline(60, color="gray", ls=":", lw=2, alpha=0.5, label="checkpoint")
ax.set_xlabel("time [s]")
ax.set_ylabel(r"$x_1(t)$")
ax.set_title("Checkpoint Rollback: Three Futures from the Same State")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

All three scenarios start from the exact same state at t=60s. The blue continuation matches the original trajectory perfectly, confirming checkpoint fidelity. The stronger coupling (orange) produces faster energy exchange, while the decoupled system (green) oscillates independently at its natural frequency.